In [2]:
import pandas as pd
import networkx as nx
from collections import defaultdict

# 读取数据
df = pd.read_csv('F:/3version_L4/js/components/pathSelection/Path/level4_path_cell_stat_allPath_no_cycle_filtered.csv')

# 构建有向图
G = nx.from_pandas_edgelist(df, source='start', target='end', edge_attr='value', create_using=nx.DiGraph())

print("=== 数据基本信息 ===")
print(f"节点数量: {G.number_of_nodes()}")
print(f"边数量: {G.number_of_edges()}")
print()

def find_all_paths(graph, max_length=10):  # 增加到10步，或者更多
    """找出图中所有简单路径，记录所有长度>=2的路径"""
    all_paths = []
    
    # 获取所有节点
    nodes = list(graph.nodes())
    
    for start_node in nodes:
        def dfs(current_node, path):
            # 如果路径长度>=2，记录这条路径
            if len(path) >= 2:
                # 计算路径权重
                path_weight = 0
                for i in range(len(path)-1):
                    if graph.has_edge(path[i], path[i+1]):
                        path_weight += graph[path[i]][path[i+1]]['value']
                all_paths.append((path.copy(), path_weight))
            
            # 如果达到最大长度，停止扩展
            if len(path) >= max_length:
                return
                
            # 获取当前节点的所有邻居
            neighbors = list(graph.successors(current_node))
            
            # 继续扩展路径
            for neighbor in neighbors:
                if neighbor not in path:  # 避免循环
                    dfs(neighbor, path + [neighbor])
        
        # 从每个节点开始DFS
        dfs(start_node, [start_node])
    
    return all_paths

def remove_subpaths(paths):
    """移除被更长路径包含的子路径"""
    # 按路径长度降序排序
    sorted_paths = sorted(paths, key=lambda x: len(x[0]), reverse=True)
    filtered_paths = []
    
    for current_path, current_weight in sorted_paths:
        is_subpath = False
        
        # 检查当前路径是否是已有路径的子路径
        for existing_path, existing_weight in filtered_paths:
            if is_subpath_of(current_path, existing_path):
                is_subpath = True
                break
        
        if not is_subpath:
            filtered_paths.append((current_path, current_weight))
    
    return filtered_paths

def is_subpath_of(short_path, long_path):
    """检查短路径是否是长路径的连续子序列"""
    if len(short_path) >= len(long_path):
        return False
    
    for i in range(len(long_path) - len(short_path) + 1):
        if long_path[i:i+len(short_path)] == short_path:
            return True
    return False

print("=== 正在寻找所有路径... ===")
all_paths = find_all_paths(G, max_length=15)  # 设置更大的上限

print(f"原始找到 {len(all_paths)} 条路径")

# 统计路径长度分布
length_stats = defaultdict(int)
for path, weight in all_paths:
    length_stats[len(path)] += 1

print("=== 路径长度分布 ===")
for length in sorted(length_stats.keys()):
    print(f"{length}步路径: {length_stats[length]} 条")
print()

# 按路径长度分组
paths_by_length = defaultdict(list)
for path, weight in all_paths:
    paths_by_length[len(path)].append((path, weight))

# 只对包含相同起点的路径进行子路径过滤
print("=== 移除子路径... ===")
all_filtered_paths = []

# 按起点分组处理
paths_by_start = defaultdict(list)
for path, weight in all_paths:
    start_node = path[0]
    paths_by_start[start_node].append((path, weight))

# 对每个起点的路径进行子路径过滤
for start_node, paths in paths_by_start.items():
    filtered = remove_subpaths(paths)
    all_filtered_paths.extend(filtered)

# 重新按长度分组
filtered_paths_by_length = defaultdict(list)
for path, weight in all_filtered_paths:
    filtered_paths_by_length[len(path)].append((path, weight))

total_filtered = len(all_filtered_paths)
print(f"过滤后剩余 {total_filtered} 条路径")

# 统计过滤后的路径长度分布
print("=== 过滤后路径长度分布 ===")
for length in sorted(filtered_paths_by_length.keys()):
    print(f"{length}步路径: {len(filtered_paths_by_length[length])} 条")
print()

# 显示每种长度的路径
for length in sorted(filtered_paths_by_length.keys()):
    paths = filtered_paths_by_length[length]
    print(f"=== {length}步路径 ({len(paths)}条) ===")
    
    # 按权重排序，显示前20条
    sorted_paths = sorted(paths, key=lambda x: x[1], reverse=True)
    
    for i, (path, weight) in enumerate(sorted_paths[:20]):
        path_str = " -> ".join(path)
        print(f"{i+1:2d}. {path_str} (权重: {weight})")
    
    if len(sorted_paths) > 20:
        print(f"    ... 还有 {len(sorted_paths)-20} 条路径")
    print()

# 特别关注Brain相关的最长路径
print("=== Brain相关最长路径 ===")
brain_paths = []
for path, weight in all_filtered_paths:
    if any('Brain' in node for node in path):
        brain_paths.append((path, weight))

# 按长度降序排序，找最长的Brain路径
brain_paths.sort(key=lambda x: (len(x[0]), x[1]), reverse=True)

max_brain_length = max(len(path) for path, weight in brain_paths) if brain_paths else 0
print(f"Brain相关路径最大长度: {max_brain_length} 步")

print("最长的Brain路径（前10条）:")
for i, (path, weight) in enumerate(brain_paths[:10]):
    path_str = " -> ".join(path)
    print(f"{i+1:2d}. [{len(path)}步] {path_str} (权重: {weight})")

# 按长度分组显示Brain路径
print("\n=== Brain路径按长度分组 ===")
brain_paths_by_length = defaultdict(list)
for path, weight in brain_paths:
    brain_paths_by_length[len(path)].append((path, weight))

for length in sorted(brain_paths_by_length.keys(), reverse=True)[:5]:  # 只显示最长的5种长度
    paths = brain_paths_by_length[length]
    print(f"\n--- Brain {length}步路径 ({len(paths)}条) ---")
    sorted_brain_paths = sorted(paths, key=lambda x: x[1], reverse=True)
    
    for i, (path, weight) in enumerate(sorted_brain_paths[:10]):
        path_str = " -> ".join(path)
        print(f"{i+1:2d}. {path_str} (权重: {weight})")
    
    if len(sorted_brain_paths) > 10:
        print(f"    ... 还有 {len(sorted_brain_paths)-10} 条路径")

=== 数据基本信息 ===
节点数量: 32
边数量: 31

=== 正在寻找所有路径... ===
原始找到 120 条路径
=== 路径长度分布 ===
2步路径: 31 条
3步路径: 28 条
4步路径: 23 条
5步路径: 16 条
6步路径: 8 条
7步路径: 6 条
8步路径: 5 条
9步路径: 3 条

=== 移除子路径... ===
过滤后剩余 72 条路径
=== 过滤后路径长度分布 ===
2步路径: 15 条
3步路径: 15 条
4步路径: 14 条
5步路径: 12 条
6步路径: 5 条
7步路径: 4 条
8步路径: 4 条
9步路径: 3 条

=== 2步路径 (15条) ===
 1. Heart_1_18 -> Connective tissue_1_30 (权重: 373)
 2. Brain_1_6 -> Brain_1_0 (权重: 367)
 3. Brain_1_6 -> Brain_1_4 (权重: 367)
 4. Brain_1_6 -> Notochord_1_5 (权重: 367)
 5. Cavity_1_19 -> Connective tissue_1_31 (权重: 190)
 6. AGM_1_29 -> Dermomyotome_1_28 (权重: 177)
 7. Cavity_1_12 -> Cavity_1_9 (权重: 168)
 8. AGM_1_24 -> Dermomyotome_1_26 (权重: 151)
 9. Neural crest_1_25 -> Mesenchyme_1_27 (权重: 144)
10. Neural crest_1_3 -> Brain_1_1 (权重: 120)
11. Neural crest_1_3 -> Neural crest_1_2 (权重: 120)
12. Cavity_1_10 -> Cavity_1_11 (权重: 56)
13. Cavity_1_20 -> Cavity_1_21 (权重: 51)
14. Cavity_1_20 -> Cavity_1_22 (权重: 51)
15. Cavity_1_20 -> Mesenchyme_1_23 (权重: 51)

=== 3步路径 (15条) ===
 1. He

In [3]:
def export_paths_csv(all_filtered_paths):
    """导出CSV格式的路径数据"""
    
    # 路径概要
    paths_summary = []
    for i, (path, weight) in enumerate(all_filtered_paths):
        paths_summary.append({
            'path_id': i,
            'length': len(path),
            'weight': weight,
            'start_node': path[0],
            'end_node': path[-1],
            'path_string': ' -> '.join(path),
        })
  
    # 保存CSV
    pd.DataFrame(paths_summary).to_csv('paths_summary_KJ.csv', index=False)
    
    print("✅ CSV文件已保存: paths_summary_KJ.csv")

# 导出CSV
export_paths_csv(all_filtered_paths)

✅ CSV文件已保存: paths_summary_KJ.csv


In [4]:
import pandas as pd
import networkx as nx
from collections import defaultdict

# 读取数据
df = pd.read_csv('F:/2version/js/components/pathSelection/Path/level6_path_cell_stat_allPath_no_cycle_filtered.csv')

# 构建有向图
G = nx.from_pandas_edgelist(df, source='start', target='end', edge_attr='value', create_using=nx.DiGraph())

print("=== 数据基本信息 ===")
print(f"节点数量: {G.number_of_nodes()}")
print(f"边数量: {G.number_of_edges()}")
print()

def find_all_paths(graph, max_length=10):
    """找出图中所有简单路径，记录所有长度>=2的路径"""
    all_paths = []
    
    # 获取所有节点
    nodes = list(graph.nodes())
    
    def dfs(current_node, path, path_weight):
        # 如果路径长度>=2，记录这条路径
        if len(path) >= 2:
            all_paths.append((path.copy(), path_weight))
        
        # 如果达到最大长度，停止扩展
        if len(path) >= max_length:
            return
        
        # 获取当前节点的所有邻居
        neighbors = list(graph.successors(current_node))
        
        # 继续扩展路径
        for neighbor in neighbors:
            if neighbor not in path:  # 避免循环
                new_weight = path_weight + graph[current_node][neighbor]['value']
                dfs(neighbor, path + [neighbor], new_weight)
    
    # 从每个节点开始DFS
    for start_node in nodes:
        dfs(start_node, [start_node], 0)
    
    return all_paths

def remove_subpaths(paths):
    """移除被更长路径包含的子路径"""
    # 按路径长度降序排序
    sorted_paths = sorted(paths, key=lambda x: len(x[0]), reverse=True)
    filtered_paths = []
    
    for current_path, current_weight in sorted_paths:
        is_subpath = False
        
        # 检查当前路径是否是已有路径的子路径
        for existing_path, existing_weight in filtered_paths:
            if is_subpath_of(current_path, existing_path):
                is_subpath = True
                break
        
        if not is_subpath:
            filtered_paths.append((current_path, current_weight))
    
    return filtered_paths

def is_subpath_of(short_path, long_path):
    """检查短路径是否是长路径的连续子序列"""
    if len(short_path) >= len(long_path):
        return False
    
    for i in range(len(long_path) - len(short_path) + 1):
        if long_path[i:i+len(short_path)] == short_path:
            return True
    return False

print("=== 正在寻找所有路径... ===")
all_paths = find_all_paths(G, max_length=15)  # 设置更大的上限

print(f"原始找到 {len(all_paths)} 条路径")

# 统计路径长度分布
length_stats = defaultdict(int)
for path, weight in all_paths:
    length_stats[len(path)] += 1

print("=== 路径长度分布 ===")
for length in sorted(length_stats.keys()):
    print(f"{length}步路径: {length_stats[length]} 条")
print()

# 移除子路径
print("=== 移除子路径... ===")
filtered_paths = remove_subpaths(all_paths)
total_filtered = len(filtered_paths)
print(f"过滤后剩余 {total_filtered} 条路径")

# 统计过滤后的路径长度分布
filtered_length_stats = defaultdict(int)
for path, weight in filtered_paths:
    filtered_length_stats[len(path)] += 1

print("=== 过滤后路径长度分布 ===")
for length in sorted(filtered_length_stats.keys()):
    print(f"{length}步路径: {filtered_length_stats[length]} 条")
print()

# 将过滤后的路径保存到CSV文件
output_df = pd.DataFrame(filtered_paths, columns=['path', 'weight'])
output_df['path'] = output_df['path'].apply(lambda x: ' -> '.join(x))  # 将路径列表转换为字符串
output_df.to_csv('filtered_paths.csv', index=False)

print("=== 过滤后的路径已保存到 filtered_paths.csv ===")

=== 数据基本信息 ===
节点数量: 104
边数量: 105

=== 正在寻找所有路径... ===
原始找到 1095 条路径
=== 路径长度分布 ===
2步路径: 105 条
3步路径: 104 条
4步路径: 104 条
5步路径: 107 条
6步路径: 103 条
7步路径: 101 条
8步路径: 100 条
9步路径: 91 条
10步路径: 76 条
11步路径: 61 条
12步路径: 52 条
13步路径: 42 条
14步路径: 29 条
15步路径: 20 条

=== 移除子路径... ===
过滤后剩余 91 条路径
=== 过滤后路径长度分布 ===
2步路径: 6 条
3步路径: 6 条
4步路径: 3 条
5步路径: 9 条
6步路径: 6 条
8步路径: 3 条
9步路径: 8 条
10步路径: 9 条
11步路径: 5 条
12步路径: 5 条
13步路径: 6 条
14步路径: 5 条
15步路径: 20 条

=== 过滤后的路径已保存到 filtered_paths.csv ===


In [6]:
import pandas as pd

# 读取数据
df = pd.read_csv('./KJ/cell_annotation_all.csv')  # 请根据实际文件路径调整

print("=== 数据基本信息 ===")
print(f"总记录数: {len(df)}")
print(f"细胞类型数: {df['annotation'].nunique()}")
print(f"时间点数: {df['time'].nunique()}")
print()

# 统计各细胞类型在不同时间点的数量
cell_time_counts = df.groupby(['annotation', 'time']).size().unstack(fill_value=0)
print("=== 各细胞类型在不同时间点的分布 ===")
print(cell_time_counts)
print()

# 计算总数和占比
cell_totals = df['annotation'].value_counts().sort_index()
total_cells = len(df)

print("=== 细胞类型统计 ===")
for cell_type in cell_totals.index:
    count = cell_totals[cell_type]
    ratio = count / total_cells
    print(f"{cell_type}: {count} 个细胞 ({ratio:.3f})")
print()

# 准备输出数据
output_data = []

for cell_type in cell_time_counts.index:
    row_data = {'annotation': cell_type}
    
    # 添加各时间点的数量
    time_columns = []
    for time_point in sorted(cell_time_counts.columns):
        time_col = f'time{time_point}'
        time_columns.append(time_col)
        row_data[time_col] = cell_time_counts.loc[cell_type, time_point]
    
    # 计算总数和占比
    total_count = cell_totals[cell_type]
    ratio = total_count / total_cells
    
    row_data['total'] = total_count
    row_data['ratio'] = round(ratio, 3)
    
    output_data.append(row_data)

# 创建输出DataFrame
output_df = pd.DataFrame(output_data)

# 确保列顺序正确
columns_order = ['annotation'] + time_columns + ['total', 'ratio']
output_df = output_df[columns_order]

print("=== 输出数据预览 ===")
print(output_df)
print()

# 保存到CSV文件
output_file = 'F:/2version/js/components/pathSelection/Path/cellcount_KJ.csv'
output_df.to_csv(output_file, index=False)

print(f"✅ 数据已保存到: {output_file}")

# 验证保存的文件
print("\n=== 验证保存的文件 ===")
saved_df = pd.read_csv(output_file)
print(saved_df)

# 额外统计信息
print("\n=== 额外统计信息 ===")
print("时间点分布:")
time_distribution = df['time'].value_counts().sort_index()
for time_point, count in time_distribution.items():
    print(f"  时间点{time_point}: {count} 个细胞")

print(f"\n总细胞数: {total_cells}")
print(f"平均每个时间点细胞数: {total_cells / df['time'].nunique():.1f}")

# 检查占比总和
total_ratio = output_df['ratio'].sum()
print(f"占比总和: {total_ratio:.3f} (应该接近1.0)")

=== 数据基本信息 ===
总记录数: 5913
细胞类型数: 12
时间点数: 1

=== 各细胞类型在不同时间点的分布 ===
time                  1
annotation             
AGM                 452
Brain              1518
Branchial arch      200
Cavity              882
Connective tissue   283
Dermomyotome        273
Heart               382
Liver                98
Mesenchyme          337
Neural crest       1008
Notochord           236
Sclerotome          244

=== 细胞类型统计 ===
AGM: 452 个细胞 (0.076)
Brain: 1518 个细胞 (0.257)
Branchial arch: 200 个细胞 (0.034)
Cavity: 882 个细胞 (0.149)
Connective tissue: 283 个细胞 (0.048)
Dermomyotome: 273 个细胞 (0.046)
Heart: 382 个细胞 (0.065)
Liver: 98 个细胞 (0.017)
Mesenchyme: 337 个细胞 (0.057)
Neural crest: 1008 个细胞 (0.170)
Notochord: 236 个细胞 (0.040)
Sclerotome: 244 个细胞 (0.041)

=== 输出数据预览 ===
           annotation  time1  total  ratio
0                 AGM    452    452  0.076
1               Brain   1518   1518  0.257
2      Branchial arch    200    200  0.034
3              Cavity    882    882  0.149
4   Connective tissue   

## Cavity_1_44 -> Brain_1_42 -> Brain_1_43 -> Brain_1_33 -> Brain_1_34 -> Cavity_1_68 (权重: 626)

这个代码可以找出某一个细胞他的周围细胞是什么，输出一个csv


cell_type,cell_num,x,y


Brain,714,122.5,230.6


Cavity,405,121.3,233.5


Notochord,184,116.0,222.5


Neural crest,104,110.3,224.3


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.neighbors import BallTree
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Circle
import matplotlib.patches as mpatches
from collections import defaultdict

# 设置要分析的 embedding_level
embedding_level = 6
target_cavity_type = "Brain_1_33"

# 读取数据
annotation_df = pd.read_csv('./cell_annotation_all.csv')
embedding_df = pd.read_csv('./cell_embedding_info.csv')

# 合并数据
merged_df = pd.merge(
    embedding_df[['cell_index', 'embedding_level', 'embedding_index']],
    annotation_df[['cell_index', 'annotation', 'x', 'y', 'time']],
    on='cell_index'
)

# 构建完整的细胞类型标识（annotation_time_embedding）
merged_df['annotation_time_embedding'] = (
    merged_df['annotation'] + '_' + 
    merged_df['time'].astype(int).astype(str) + '_' + 
    merged_df['embedding_index'].astype(str)
)

# 筛选指定embedding_level的数据
level_data = merged_df[merged_df['embedding_level'] == embedding_level]

# 修正Y坐标 - 取绝对值
level_data['y'] = level_data['y'].abs()

print(f"=== 查找{target_cavity_type}类型细胞及其邻居DBSCAN聚类分析 ===")
print(f"Level {embedding_level}")

# 1. 找到所有目标类型的细胞
cavity_cells = level_data[level_data['annotation_time_embedding'] == target_cavity_type]

print(f"找到 {len(cavity_cells)} 个{target_cavity_type}细胞")

if len(cavity_cells) == 0:
    print(f"未找到{target_cavity_type}细胞")
    print("数据中存在的相关类型：")
    target_base = target_cavity_type.split('_')[0]
    related_cells = level_data[level_data['annotation_time_embedding'].str.contains(target_base, case=False, na=False)]['annotation_time_embedding'].unique()
    for cell_type in sorted(related_cells):
        count = len(level_data[level_data['annotation_time_embedding'] == cell_type])
        print(f"  {cell_type}: {count} 个细胞")
else:
    # 2. 计算目标细胞的重心
    cavity_centroid_x = cavity_cells['x'].mean()
    cavity_centroid_y = cavity_cells['y'].mean()
    cavity_count = len(cavity_cells)
    
    print(f"{target_cavity_type}细胞重心: ({cavity_centroid_x:.1f}, {cavity_centroid_y:.1f})")
    
    # 3. 使用BallTree找到目标细胞的邻居
    coordinates = level_data[['x', 'y']].values
    tree = BallTree(coordinates, metric='euclidean')
    cavity_coords = cavity_cells[['x', 'y']].values
    neighbor_indices_list = tree.query_radius(cavity_coords, r=20)  # 扩大搜索半径
    
    # 合并所有邻居索引，去重
    all_neighbor_indices = set()
    for neighbor_indices in neighbor_indices_list:
        all_neighbor_indices.update(neighbor_indices)
    
    neighbor_cell_indices = [level_data.index[i] for i in all_neighbor_indices]
    neighbor_cells = level_data.loc[neighbor_cell_indices]
    
    print(f"找到 {len(neighbor_cells)} 个邻居细胞（包括{target_cavity_type}细胞本身）")
    
    # 4. 按annotation类型分组，对每种类型进行DBSCAN聚类，并分析每个簇
    all_clusters = []  # 存储所有簇的信息
    annotation_clusters = {}
    
    print(f"\n=== 使用DBSCAN分析各种细胞类型的聚类簇 ===")
    
    # 获取目标细胞的基本类型（去掉time和embedding信息）
    target_base_annotation = target_cavity_type.split('_')[0]
    
    for annotation in neighbor_cells['annotation'].unique():
        if annotation == target_base_annotation:
            continue  # 跳过目标细胞本身
            
        annotation_cells = neighbor_cells[neighbor_cells['annotation'] == annotation]
        
        if len(annotation_cells) < 5:  # 细胞数量太少，跳过
            continue
            
        print(f"\n--- {annotation} ({len(annotation_cells)} 个细胞) ---")
        
        # 准备DBSCAN数据
        coords = annotation_cells[['x', 'y']].values
        cell_indices = annotation_cells.index.values
        
        # 标准化坐标
        scaler = StandardScaler()
        coords_scaled = scaler.fit_transform(coords)
        
        # 应用DBSCAN聚类 - 调整参数以获得更好的聚类效果
        eps_value = 0.4
        min_samples = max(3, len(annotation_cells) // 10)  # 动态调整min_samples
        dbscan = DBSCAN(eps=eps_value, min_samples=min_samples)
        cluster_labels = dbscan.fit_predict(coords_scaled)
        
        # 分析聚类结果
        n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
        n_noise = list(cluster_labels).count(-1)
        
        print(f"  DBSCAN结果: {n_clusters} 个聚类, {n_noise} 个噪声点")
        
        # 存储聚类信息
        annotation_clusters[annotation] = {
            'coords': coords,
            'labels': cluster_labels,
            'cell_indices': cell_indices,
            'n_clusters': n_clusters,
            'n_noise': n_noise
        }
        
        # 分析每个聚类簇
        for cluster_id in set(cluster_labels):
            if cluster_id == -1:  # 跳过噪声点
                continue
                
            # 获取该簇的细胞
            cluster_mask = cluster_labels == cluster_id
            cluster_coords = coords[cluster_mask]
            cluster_cell_count = len(cluster_coords)
            
            if cluster_cell_count < 3:  # 跳过过小的簇
                continue
            
            # 计算簇的重心
            cluster_centroid_x = cluster_coords[:, 0].mean()
            cluster_centroid_y = cluster_coords[:, 1].mean()
            
            # 计算到目标细胞的距离
            distance_to_target = np.sqrt((cluster_centroid_x - cavity_centroid_x)**2 + 
                                       (cluster_centroid_y - cavity_centroid_y)**2)
            
            # 计算簇的紧密度（簇内细胞间的平均距离）
            if cluster_cell_count > 1:
                intra_distances = []
                for i in range(len(cluster_coords)):
                    for j in range(i+1, len(cluster_coords)):
                        dist = np.sqrt(np.sum((cluster_coords[i] - cluster_coords[j])**2))
                        intra_distances.append(dist)
                cluster_compactness = np.mean(intra_distances) if intra_distances else 0
            else:
                cluster_compactness = 0
            
            # 存储簇信息
            cluster_info = {
                'annotation': annotation,
                'cluster_id': cluster_id,
                'centroid_x': cluster_centroid_x,
                'centroid_y': cluster_centroid_y,
                'cell_count': cluster_cell_count,
                'distance_to_target': distance_to_target,
                'compactness': cluster_compactness,
                'coords': cluster_coords
            }
            
            all_clusters.append(cluster_info)
            
            print(f"    簇 {cluster_id}: 重心({cluster_centroid_x:.1f}, {cluster_centroid_y:.1f}), "
                  f"{cluster_cell_count}个细胞, 距{target_base_annotation} {distance_to_target:.1f}, "
                  f"紧密度 {cluster_compactness:.2f}")
    
    # 5. 生成传统格式的邻居统计数据（兼容原有格式）
    neighbor_stats = []
    
    # 添加目标细胞本身
    neighbor_stats.append({
        'cell_type': target_base_annotation,
        'cell_num': cavity_count,
        'x': round(cavity_centroid_x, 1),
        'y': round(cavity_centroid_y, 1)
    })
    
    # 为每种细胞类型添加统计信息（合并所有簇）
    annotation_summary = {}
    for cluster in all_clusters:
        annotation = cluster['annotation']
        if annotation not in annotation_summary:
            annotation_summary[annotation] = {
                'total_cells': 0,
                'weighted_x': 0,
                'weighted_y': 0
            }
        
        # 按细胞数量加权计算位置
        weight = cluster['cell_count']
        annotation_summary[annotation]['total_cells'] += weight
        annotation_summary[annotation]['weighted_x'] += cluster['centroid_x'] * weight
        annotation_summary[annotation]['weighted_y'] += cluster['centroid_y'] * weight
    
    # 计算每种类型的加权重心
    for annotation, summary in annotation_summary.items():
        if summary['total_cells'] > 0:
            avg_x = summary['weighted_x'] / summary['total_cells']
            avg_y = summary['weighted_y'] / summary['total_cells']
            
            neighbor_stats.append({
                'cell_type': annotation,
                'cell_num': summary['total_cells'],
                'x': round(avg_x, 1),
                'y': round(avg_y, 1)
            })
            
            print(f"{annotation}: {summary['total_cells']} 个细胞, 加权重心位置: ({avg_x:.1f}, {avg_y:.1f})")
    
    # 6. 创建DataFrame并保存到CSV
    neighbor_stats_df = pd.DataFrame(neighbor_stats)
    neighbor_stats_df = neighbor_stats_df.sort_values('cell_num', ascending=False)
    
    # 保存到CSV文件
    output_filename = f'{target_cavity_type}.csv'
    neighbor_stats_df.to_csv(output_filename, index=False)
    
    print(f"\n=== 邻居细胞统计结果已保存到 {output_filename} ===")
    print("文件内容预览:")
    print(neighbor_stats_df.to_string(index=False))
    
    # 7. 输出详细的DBSCAN聚类分析结果
    print(f"\n=== DBSCAN聚类详细分析结果 ===")
    print(f"总共发现 {len(all_clusters)} 个有效聚类簇")
    
    # 按距离目标细胞排序
    all_clusters_sorted = sorted(all_clusters, key=lambda x: x['distance_to_target'])
    
    print(f"\n按距离{target_base_annotation}排序的簇信息:")
    for i, cluster in enumerate(all_clusters_sorted, 1):
        print(f"{i:2d}. {cluster['annotation']} - 簇{cluster['cluster_id']}")
        print(f"     重心位置: ({cluster['centroid_x']:.1f}, {cluster['centroid_y']:.1f})")
        print(f"     细胞数量: {cluster['cell_count']} 个")
        print(f"     距{target_base_annotation}距离: {cluster['distance_to_target']:.1f}")
        print(f"     簇紧密度: {cluster['compactness']:.2f}")
        print()
    
    # 按细胞类型统计簇信息
    print(f"\n按细胞类型统计的簇信息:")
    annotation_cluster_stats = {}
    for cluster in all_clusters:
        annotation = cluster['annotation']
        if annotation not in annotation_cluster_stats:
            annotation_cluster_stats[annotation] = {
                'cluster_count': 0,
                'total_cells': 0,
                'avg_distance': 0,
                'min_distance': float('inf'),
                'max_distance': 0
            }
        
        stats = annotation_cluster_stats[annotation]
        stats['cluster_count'] += 1
        stats['total_cells'] += cluster['cell_count']
        stats['avg_distance'] += cluster['distance_to_target']
        stats['min_distance'] = min(stats['min_distance'], cluster['distance_to_target'])
        stats['max_distance'] = max(stats['max_distance'], cluster['distance_to_target'])
    
    # 计算平均距离
    for annotation, stats in annotation_cluster_stats.items():
        stats['avg_distance'] /= stats['cluster_count']
    
    for annotation, stats in annotation_cluster_stats.items():
        print(f"{annotation}:")
        print(f"  簇数量: {stats['cluster_count']}")
        print(f"  总细胞数: {stats['total_cells']}")
        print(f"  平均距离{target_base_annotation}: {stats['avg_distance']:.1f}")
        print(f"  距离范围: {stats['min_distance']:.1f} - {stats['max_distance']:.1f}")
        print()
    
    # 8. 输出最终统计信息
    print(f"\n=== 详细统计信息 ===")
    print(f"目标细胞类型: {target_cavity_type}")
    print(f"目标细胞数量: {cavity_count}")
    print(f"目标细胞重心: ({cavity_centroid_x:.1f}, {cavity_centroid_y:.1f})")
    print(f"搜索半径: 20")
    print(f"邻居细胞总数: {len(neighbor_cells)}")
    print(f"有效聚类簇数: {len(all_clusters)}")
    print(f"邻居细胞类型数: {len(annotation_summary)}")
    print(f"\n各类型细胞数量占比:")
    total_neighbors = len(neighbor_cells)
    for _, row in neighbor_stats_df.iterrows():
        percentage = row['cell_num'] / total_neighbors * 100
        print(f"  {row['cell_type']}: {row['cell_num']} 个 ({percentage:.1f}%) - 重心({row['x']}, {row['y']})")

=== 查找Cavity_1_44类型细胞及其邻居分析 ===
Level 6
找到 191 个Brain_1_33细胞
Brain_1_33细胞重心: (110.2, 230.1)
找到 2668 个邻居细胞（包括Brain_1_33细胞本身）
Neural crest: 563 个细胞, 重心位置: (100.6, 215.2)
Brain: 1375 个细胞, 重心位置: (111.7, 225.1)
Cavity: 492 个细胞, 重心位置: (116.5, 231.2)
Branchial arch: 15 个细胞, 重心位置: (114.1, 207.8)
Notochord: 222 个细胞, 重心位置: (117.1, 220.3)
Sclerotome: 1 个细胞, 重心位置: (108.0, 210.0)

=== 邻居细胞统计结果已保存到 Brain_1_33.csv ===
文件内容预览:
     cell_type  cell_num     x     y
         Brain      1375 111.7 225.1
  Neural crest       563 100.6 215.2
        Cavity       492 116.5 231.2
     Notochord       222 117.1 220.3
    Brain_1_33       191 110.2 230.1
Branchial arch        15 114.1 207.8
    Sclerotome         1 108.0 210.0

=== 详细统计信息 ===
目标细胞类型: Brain_1_33
目标细胞数量: 191
目标细胞重心: (110.2, 230.1)
搜索半径: 10
邻居细胞总数: 2668
邻居细胞类型数: 12

各类型细胞数量占比:
  Brain: 1375 个 (51.5%) - 重心(111.7, 225.1)
  Neural crest: 563 个 (21.1%) - 重心(100.6, 215.2)
  Cavity: 492 个 (18.4%) - 重心(116.5, 231.2)
  Notochord: 222 个 (8.3%) - 重心(117.1, 

C:\Users\Windows\AppData\Local\Temp\ipykernel_27976\562064310.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  level_data['y'] = level_data['y'].abs()


# 找出通讯强度以及top10通讯


这个代码生成了三个文件，例如
A_as_Sent.csv"
A_as_receive.csv"
A_total.csv"

In [10]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ==================== 配置区域 ====================
# 只需要修改这里的参数
TARGET_CELL = 'Cavity'  # 目标细胞类型名称
GROUP_NAME = 'Cavity_1_68'  # 群体名称
BASE_PATH = "F:/allCellChat_level6/"  # 基础路径
TOP_N = 10  # 显示前N个通道

# ==================== 核心分析类 ====================
class CellPhoneDBAnalyzer:
    def __init__(self, group_name, base_path="F:/allCellChat_level6/"):
        self.group_name = group_name
        self.base_path = base_path
        self.results_dir = os.path.join(base_path, group_name)
        self.significant_means = None
        self.pvalues = None
        self.cell_pair_columns = []
        self._load_data()
        
    def _load_data(self):
        """加载CellPhoneDB结果文件"""
        if not os.path.exists(self.results_dir):
            print(f"错误：目录 {self.results_dir} 不存在！")
            return
        
        # 查找文件
        for file in os.listdir(self.results_dir):
            file_lower = file.lower()
            if 'significant_means' in file_lower and file.endswith('.txt'):
                self.significant_means = pd.read_csv(os.path.join(self.results_dir, file), sep='\t')
            elif 'pvalue' in file_lower and file.endswith('.txt'):
                self.pvalues = pd.read_csv(os.path.join(self.results_dir, file), sep='\t')
        
        # 获取细胞类型对列
        if self.significant_means is not None:
            self.cell_pair_columns = [col for col in self.significant_means.columns 
                                    if '|' in col and col != 'interacting_pair']
    
    def analyze_cell_channels(self, target_cell, top_n=10):
        """分析指定细胞类型的通讯通道"""
        if self.significant_means is None:
            print("没有找到significant_means数据文件")
            return None
        
        sender_channels = []
        receiver_channels = []
        
        # 分析每个交互对
        for _, row in self.significant_means.iterrows():
            interaction_pair = row['interacting_pair']
            
            # 获取对应的p值行
            pvalue_row = None
            if self.pvalues is not None:
                pvalue_match = self.pvalues[self.pvalues['interacting_pair'] == interaction_pair]
                if not pvalue_match.empty:
                    pvalue_row = pvalue_match.iloc[0]
            
            # 解析配体和受体
            ligand, receptor = "", ""
            if '_' in interaction_pair:
                parts = interaction_pair.split('_')
                if len(parts) >= 2:
                    ligand, receptor = parts[0], parts[1]
            
            # 检查每个细胞类型对
            for col in self.cell_pair_columns:
                if col in self.significant_means.columns:
                    score = row[col]
                    
                    if pd.notna(score) and score > 0:
                        cell_types = col.split('|')
                        if len(cell_types) == 2:
                            sender, receiver_cell = cell_types
                            
                            # 获取p值
                            pvalue = None
                            if pvalue_row is not None and col in pvalue_row.index:
                                pvalue = pvalue_row[col]
                            
                            # 格式化p值显示
                            if pd.notna(pvalue):
                                if pvalue < 0.001:
                                    p_display = "<0.001"
                                elif pvalue < 0.01:
                                    p_display = f"{pvalue:.3f}"
                                else:
                                    p_display = f"{pvalue:.3f}"
                            else:
                                p_display = 'N/A'
                            
                            channel_info = {
                                '通道': f"{sender}→{receiver_cell}",
                                '接收方': receiver_cell,
                                '接收方基因': receptor,
                                '发送方': sender,
                                '发送方基因': ligand,
                                '强度': score,
                                '显著性': p_display
                            }
                            
                            # 检查是否为目标细胞
                            if sender == target_cell:
                                sender_channels.append(channel_info)
                            if receiver_cell == target_cell:
                                receiver_channels.append(channel_info)
        
        # 转换为DataFrame并排序
        sender_df = pd.DataFrame(sender_channels)
        receiver_df = pd.DataFrame(receiver_channels)
        
        if not sender_df.empty:
            sender_df = sender_df.sort_values('强度', ascending=False).head(top_n)
        if not receiver_df.empty:
            receiver_df = receiver_df.sort_values('强度', ascending=False).head(top_n)
        
        # 打印结果
        self._print_channels(sender_df, f"{target_cell} 作为发送方的最强{top_n}个通道")
        self._print_channels(receiver_df, f"{target_cell} 作为接收方的最强{top_n}个通道")
        
        return {
            'sender_channels': sender_df,
            'receiver_channels': receiver_df
        }
    
    def _print_channels(self, df, title):
        """打印通道信息"""
        if df.empty:
            print(f"\n没有找到相关通道数据")
            return
        
        print(f"\n=== {title} ===")
        print("=" * 100)
        print(f"{'通道':<25} {'接收方':<15} {'接收方基因':<12} {'发送方':<15} {'发送方基因':<12} {'强度':<8} {'显著性':<8}")
        print("=" * 100)
        
        for _, row in df.iterrows():
            print(f"{row['通道']:<25} {row['接收方']:<15} {row['接收方基因']:<12} "
                  f"{row['发送方']:<15} {row['发送方基因']:<12} "
                  f"{row['强度']:<8.3f} {row['显著性']:<8}")

    def calculate_neighbor_intensities(self, target_cell):
        """计算邻居细胞与目标细胞的通讯强度统计"""
        if self.significant_means is None:
            print("没有找到significant_means数据文件")
            return None
        
        # 存储各种统计数据
        send_stats = {}  # target_cell发送给其他细胞的强度
        receive_stats = {}  # 其他细胞发送给target_cell的强度
        
        # 分析每个交互对
        for _, row in self.significant_means.iterrows():
            # 检查每个细胞类型对
            for col in self.cell_pair_columns:
                if col in self.significant_means.columns:
                    score = row[col]
                    
                    if pd.notna(score) and score > 0:
                        cell_types = col.split('|')
                        if len(cell_types) == 2:
                            sender, receiver = cell_types
                            
                            # 目标细胞作为发送方
                            if sender == target_cell and receiver != target_cell:
                                if receiver not in send_stats:
                                    send_stats[receiver] = []
                                send_stats[receiver].append(score)
                            
                            # 目标细胞作为接收方
                            elif receiver == target_cell and sender != target_cell:
                                if sender not in receive_stats:
                                    receive_stats[sender] = []
                                receive_stats[sender].append(score)
        
        # 计算统计指标
        send_summary = []
        receive_summary = []
        total_summary = []
        
        # 获取所有邻居细胞
        all_neighbors = set(send_stats.keys()) | set(receive_stats.keys())
        
        for neighbor in all_neighbors:
            # 发送强度统计
            send_scores = send_stats.get(neighbor, [])
            send_total = sum(send_scores)
            send_count = len(send_scores)
            send_avg = send_total / send_count if send_count > 0 else 0
            
            send_summary.append({
                '邻居细胞': neighbor,
                '通道数量': send_count,
                '总强度': send_total,
                '平均强度': send_avg,
                '最大强度': max(send_scores) if send_scores else 0,
                '最小强度': min(send_scores) if send_scores else 0
            })
            
            # 接收强度统计
            receive_scores = receive_stats.get(neighbor, [])
            receive_total = sum(receive_scores)
            receive_count = len(receive_scores)
            receive_avg = receive_total / receive_count if receive_count > 0 else 0
            
            receive_summary.append({
                '邻居细胞': neighbor,
                '通道数量': receive_count,
                '总强度': receive_total,
                '平均强度': receive_avg,
                '最大强度': max(receive_scores) if receive_scores else 0,
                '最小强度': min(receive_scores) if receive_scores else 0
            })
            
            # 总和统计
            total_intensity = send_total + receive_total
            total_count = send_count + receive_count
            total_avg = total_intensity / total_count if total_count > 0 else 0
            
            total_summary.append({
                '邻居细胞': neighbor,
                '发送通道数': send_count,
                '接收通道数': receive_count,
                '总通道数': total_count,
                '发送总强度': send_total,
                '接收总强度': receive_total,
                '通讯总强度': total_intensity,
                '发送平均强度': send_avg,
                '接收平均强度': receive_avg,
                '总平均强度': total_avg
            })
        
        # 转换为DataFrame并排序
        send_df = pd.DataFrame(send_summary).sort_values('总强度', ascending=False)
        receive_df = pd.DataFrame(receive_summary).sort_values('总强度', ascending=False)
        total_df = pd.DataFrame(total_summary).sort_values('通讯总强度', ascending=False)
        
        return {
            'send_stats': send_df,
            'receive_stats': receive_df,
            'total_stats': total_df
        }
    
    def generate_neighbor_top10_channels(self, target_cell, top_n=10):
        """为每个邻居细胞生成前10个通讯通道详细信息"""
        if self.significant_means is None:
            print("没有找到significant_means数据文件")
            return None
        
        # 存储每个邻居细胞的通道详细信息
        neighbor_channels = {}
        
        # 分析每个交互对
        for _, row in self.significant_means.iterrows():
            interaction_pair = row['interacting_pair']
            
            # 获取对应的p值行
            pvalue_row = None
            if self.pvalues is not None:
                pvalue_match = self.pvalues[self.pvalues['interacting_pair'] == interaction_pair]
                if not pvalue_match.empty:
                    pvalue_row = pvalue_match.iloc[0]
            
            # 解析配体和受体
            ligand, receptor = "", ""
            if '_' in interaction_pair:
                parts = interaction_pair.split('_')
                if len(parts) >= 2:
                    ligand, receptor = parts[0], parts[1]
            
            # 检查每个细胞类型对
            for col in self.cell_pair_columns:
                if col in self.significant_means.columns:
                    score = row[col]
                    
                    if pd.notna(score) and score > 0:
                        cell_types = col.split('|')
                        if len(cell_types) == 2:
                            sender, receiver_cell = cell_types
                            
                            # 获取p值
                            pvalue = None
                            if pvalue_row is not None and col in pvalue_row.index:
                                pvalue = pvalue_row[col]
                            
                            # 格式化p值显示
                            if pd.notna(pvalue):
                                if pvalue < 0.001:
                                    p_display = "<0.001"
                                elif pvalue < 0.01:
                                    p_display = f"{pvalue:.3f}"
                                else:
                                    p_display = f"{pvalue:.3f}"
                            else:
                                p_display = 'N/A'
                            
                            channel_info = {
                                '通道': f"{sender}→{receiver_cell}",
                                '接收方': receiver_cell,
                                '接收方基因': receptor,
                                '发送方': sender,
                                '发送方基因': ligand,
                                '强度': score,
                                '显著性': p_display,
                                '方向': ''  # 后续设置
                            }
                            
                            # 目标细胞作为发送方
                            if sender == target_cell and receiver_cell != target_cell:
                                if receiver_cell not in neighbor_channels:
                                    neighbor_channels[receiver_cell] = {'send': [], 'receive': []}
                                channel_info['方向'] = '发送'
                                neighbor_channels[receiver_cell]['send'].append(channel_info)
                            
                            # 目标细胞作为接收方
                            elif receiver_cell == target_cell and sender != target_cell:
                                if sender not in neighbor_channels:
                                    neighbor_channels[sender] = {'send': [], 'receive': []}
                                channel_info['方向'] = '接收'
                                neighbor_channels[sender]['receive'].append(channel_info)
        
        # 整理每个邻居的前10个通道
        all_channels = []
        
        for neighbor, channels in neighbor_channels.items():
            # 发送通道前10个
            send_channels = sorted(channels['send'], key=lambda x: x['强度'], reverse=True)[:top_n]
            for channel in send_channels:
                all_channels.append({
                    '邻居细胞': neighbor,
                    '方向': '发送',
                    '通道': channel['通道'],
                    '接收方': channel['接收方'],
                    '接收方基因': channel['接收方基因'],
                    '发送方': channel['发送方'],
                    '发送方基因': channel['发送方基因'],
                    '强度': channel['强度'],
                    '显著性': channel['显著性']
                })
            
            # 接收通道前10个
            receive_channels = sorted(channels['receive'], key=lambda x: x['强度'], reverse=True)[:top_n]
            for channel in receive_channels:
                all_channels.append({
                    '邻居细胞': neighbor,
                    '方向': '接收',
                    '通道': channel['通道'],
                    '接收方': channel['接收方'],
                    '接收方基因': channel['接收方基因'],
                    '发送方': channel['发送方'],
                    '发送方基因': channel['发送方基因'],
                    '强度': channel['强度'],
                    '显著性': channel['显著性']
                })
        
        return pd.DataFrame(all_channels)
    
    def export_neighbor_analysis(self, target_cell):
        """导出邻居细胞通讯强度分析结果"""
        stats = self.calculate_neighbor_intensities(target_cell)
        
        if stats is None:
            print("无法计算邻居细胞强度统计")
            return
        
        # 定义文件名
        send_filename = f"{self.group_name}_as_Sent.csv"
        receive_filename = f"{self.group_name}_as_receive.csv"
        total_filename = f"{self.group_name}_total.csv"
        top10_filename = f"{self.group_name}_every_top_10.csv"
        
        # 生成每个邻居细胞的前10个通道详细信息
        top10_channels = self.generate_neighbor_top10_channels(target_cell, 10)
        
        # 导出CSV文件
        try:
            stats['send_stats'].to_csv(send_filename, index=False, encoding='utf-8-sig')
            stats['receive_stats'].to_csv(receive_filename, index=False, encoding='utf-8-sig')
            stats['total_stats'].to_csv(total_filename, index=False, encoding='utf-8-sig')
            
            # 导出每个邻居的前10个通道
            if top10_channels is not None and not top10_channels.empty:
                top10_channels.to_csv(top10_filename, index=False, encoding='utf-8-sig')
                print(f"  每邻居前10通道已导出: {top10_filename}")
            
            print(f"\n📊 邻居细胞通讯强度统计:")
            print(f"  发送强度统计已导出: {send_filename}")
            print(f"  接收强度统计已导出: {receive_filename}")
            print(f"  总强度统计已导出: {total_filename}")
            
            # 打印概要信息
            print(f"\n=== {target_cell} 邻居细胞通讯强度概要 ===")
            print("\n🔵 发送强度排名 (前5名):")
            print(stats['send_stats'][['邻居细胞', '通道数量', '总强度', '平均强度']].head().to_string(index=False))
            
            print(f"\n🟢 接收强度排名 (前5名):")
            print(stats['receive_stats'][['邻居细胞', '通道数量', '总强度', '平均强度']].head().to_string(index=False))
            
            print(f"\n🟡 总通讯强度排名 (前5名):")
            print(stats['total_stats'][['邻居细胞', '总通道数', '通讯总强度', '总平均强度']].head().to_string(index=False))
            
            # 显示前10通道文件的概要信息
            if top10_channels is not None and not top10_channels.empty:
                print(f"\n🔸 每邻居前10通道统计:")
                neighbor_counts = top10_channels['邻居细胞'].value_counts()
                print(f"  涉及邻居细胞数量: {len(neighbor_counts)}")
                print(f"  总通道记录数: {len(top10_channels)}")
                
                # 显示几个示例
                print("\n示例记录（前5条）:")
                sample_df = top10_channels.head()
                for _, row in sample_df.iterrows():
                    print(f"  {row['邻居细胞']}({row['方向']}): {row['通道']} - 强度:{row['强度']:.3f}")
            
        except Exception as e:
            print(f"导出文件时出错: {str(e)}")
        
        return stats

# ==================== 执行分析 ====================

# 创建分析器并执行分析
print(f"开始分析 {TARGET_CELL} 细胞的通讯通道...")
print(f"数据来源: {os.path.join(BASE_PATH, GROUP_NAME)}")
print("-" * 50)

analyzer = CellPhoneDBAnalyzer(GROUP_NAME, BASE_PATH)

# 原有的Top10通道分析
results = analyzer.analyze_cell_channels(TARGET_CELL, TOP_N)

print(f"\n✅ Top{TOP_N}通道分析完成！")
if results:
    sender_count = len(results['sender_channels']) if not results['sender_channels'].empty else 0
    receiver_count = len(results['receiver_channels']) if not results['receiver_channels'].empty else 0
    print(f"找到 {sender_count} 个发送方通道，{receiver_count} 个接收方通道")

# 新增的邻居细胞强度统计和导出
print(f"\n" + "="*60)
print("开始计算邻居细胞通讯强度统计...")

neighbor_stats = analyzer.export_neighbor_analysis(TARGET_CELL)

print(f"\n🎉 完整分析已完成！")

开始分析 Cavity 细胞的通讯通道...
数据来源: F:/allCellChat_level6/Cavity_1_68
--------------------------------------------------

=== Cavity 作为发送方的最强10个通道 ===
通道                        接收方             接收方基因        发送方             发送方基因        强度       显著性     
Cavity→Heart              Heart           IGF2R        Cavity          IGF2         1.968    <0.001  
Cavity→Liver              Liver           BSG          Cavity          PPIA         1.178    0.017   
Cavity→Dermomyotome       Dermomyotome    CADM1        Cavity          CADM1        0.754    <0.001  
Cavity→Notochord          Notochord       NECTIN3      Cavity          CADM1        0.736    0.001   
Cavity→Notochord          Notochord       NECTIN3      Cavity          NECTIN2      0.707    <0.001  
Cavity→Heart              Heart           integrin     Cavity          LAMC1        0.664    <0.001  
Cavity→Heart              Heart           integrin     Cavity          COL1A1       0.658    <0.001  
Cavity→Heart              Heart         

In [4]:
import pandas as pd
from collections import defaultdict
import re

def extract_tissue_type(node_name):
    """从节点名称中提取组织类型"""
    # 移除后缀的数字和下划线
    tissue_type = re.sub(r'_\d+_\d+$', '', node_name)
    return tissue_type

def analyze_paths(input_file, output_file):
    """分析路径数据并生成统计结果"""
    try:
        # 读取paths_summary.csv
        df = pd.read_csv(input_file)
        
        # 统计字典
        path_stats = defaultdict(list)
        
        # 处理每一行数据
        for _, row in df.iterrows():
            start_tissue = extract_tissue_type(row['start_node'])
            end_tissue = extract_tissue_type(row['end_node'])
            path_length = row['length']
            
            # 创建起点-终点对的键
            key = (start_tissue, end_tissue)
            path_stats[key].append(path_length)
        
        # 生成统计结果
        results = []
        for (start, end), lengths in path_stats.items():
            path_num = len(lengths)
            avg_length = sum(lengths) / len(lengths)
            results.append({
                'start': start,
                'end': end,
                'path_num': path_num,
                'path_length': round(avg_length, 2)
            })
        
        # 转换为DataFrame并保存
        result_df = pd.DataFrame(results)
        result_df = result_df.sort_values(['start', 'end'])
        result_df.to_csv(output_file, index=False)
        
        print(f"分析完成！结果已保存到 {output_file}")
        print(f"总共统计了 {len(results)} 种起点-终点组合")
        print("\n前几行预览:")
        print(result_df.head(10))
        
    except FileNotFoundError:
        print(f"错误: 找不到文件 {input_file}")
    except Exception as e:
        print(f"处理过程中出现错误: {e}")

if __name__ == "__main__":
    input_file = "F:/3version_L4/js/components/pathSelection/Path/paths_summary_KJ_L4.csv"
    output_file = "path_analysis_results_KJ_L4.csv"
    analyze_paths(input_file, output_file)

分析完成！结果已保存到 path_analysis_results_KJ_L4.csv
总共统计了 17 种起点-终点组合

前几行预览:
             start                end  path_num  path_length
16             AGM       Dermomyotome         3         2.33
7            Brain              Brain         2         2.00
9            Brain             Cavity         4         5.00
10           Brain         Mesenchyme         1         6.00
8            Brain          Notochord         1         2.00
13  Branchial arch  Connective tissue         2         3.50
11          Cavity             Cavity        12         3.25
15          Cavity  Connective tissue         1         2.00
12          Cavity         Mesenchyme         4         3.50
14           Heart  Connective tissue         2         2.50
